# La Grulla Blanca — AGUA HEAVY / S4 → S5 · V07 Strict Chain

Continúa **únicamente la raíz Agua** desde el checkpoint S4 validado.

Autoridad de reanudación:
- `AGUA_S4_TECHNIQUES_HEAVY_CHECKPOINT.zip`
- SHA-256 obligatorio: `684480a57418a5ff1bbf693cbc35417814b2e31ccf0af8131f145915e9b95515`
- Estado conocido al cierre de S3→S4: técnicas completas compactadas `espejo_luna`, `latigazo_marea`, `palma_ardiente`.

Selección segura:
1. Busca el checkpoint S4 en los Inputs montados del notebook.
2. Si no está montado, intenta el enlace directo del Output anterior (requiere Internet; el enlace puede expirar).
3. **No continúa** si el SHA-256 no coincide exactamente.
4. El motor decide desde `state.json` qué trabajo pendiente de Agua debe continuar; no se fuerzan fases ni técnicas manualmente.

Salida si vuelve a ceder:
`AGUA_S5_TECHNIQUES_HEAVY_CHECKPOINT.zip`

Salida si termina:
`RESULTS_AGUA_HEAVY.zip`

Sin CANON automático; los resultados vuelven a revisión humana.


In [ ]:
RUN_MODE = "NUMERIC_SEARCH"
RESUME = True
ENTRYPOINT = "agua"
EXPECTED_RESUME_SHA256 = "684480a57418a5ff1bbf693cbc35417814b2e31ccf0af8131f145915e9b95515"
EXPECTED_RESUME_BASENAME = "AGUA_S4_TECHNIQUES_HEAVY_CHECKPOINT.zip"
CHECKPOINT_URL_FALLBACK = "https://www.kaggleusercontent.com/kf/355215077/eyJhbGciOiJkaXIiLCJlbmMiOiJBMTI4Q0JDLUhTMjU2In0..Un7twTK7fojKzrsfViaLLg.q3lUebp6tvKWN6R5kGkZmIqaQBuI7LnkT5VNU8n7VkSCMd0nVQPA41aEMcybyGi4wB4rZmCHiC3GzMWKi2cQJftiDFAWZYC1g75XG6nRivqK3fKzY6Ve49cpkxm_2xXEHXAQjlAa_ZmnaEsoO221GYElzwzaam-7G128NEzgQ4DJb1c7TX3FyBQBXuCYypWqx6rSVNG3xwk2926Uu8EdoSYaOv7tyMZWstWjYWZUGwzncVl5WXgbjf311CAKvyOsESSWLiS-PB9aSw0pAaLhcWYWDLliqW96avWmfUEmI1wF4EBhZ7bLAuKOOrlmPtx3ctQZ6fgh71DpLdeKJWq0S-FVxU9cLylRh8-o3FfOqs2aT49FiWEldS16uc1g3FE1P2hTI03fQrgnTSNgarg5zMeeLXbN_RtKYoqmUrB1RlV_Q2HRx8WEwPIervxWZ1xWtEPIrIjDcyeB5UuNkVWHHr9xggktSdGdx86P0mHuBg3iXV2AIy_CNV8o8TsBS63wYn9Egg8-LqUG79eM_k9t2KzypnW4JSE-Zi5Omyhy74YbWWbgERhKqdRiBXa_4Y3uDGNm7JUH8Mf7xkzoc8CcO6mwxaFJH0FKmwtv1zcbww0.86_BysZKRYdmpgjuq0C5NQ/TECHNIQUES_CAMPAIGN_V01/AGUA_S4_TECHNIQUES_HEAVY_CHECKPOINT.zip"

print("[CONFIG READY] RUN_MODE=", RUN_MODE, "| RESUME=", RESUME, "| ENTRYPOINT=", ENTRYPOINT, flush=True)
print("[V07 STRICT CHAIN] expected checkpoint:", EXPECTED_RESUME_BASENAME, flush=True)
print("[V07 STRICT CHAIN] expected SHA256:", EXPECTED_RESUME_SHA256, flush=True)


In [ ]:
import sys, hashlib, zipfile, json, os, subprocess, shutil, base64
from pathlib import Path
sys.dont_write_bytecode = True

INPUT_ROOT = Path("/kaggle/input")
WORKING = Path("/kaggle/working")
WORKING.mkdir(parents=True, exist_ok=True)

OFFICIAL_REPO = "https://github.com/Shein25/La-Grulla-Blanca.git"
ASSET_COMMIT = "ac184bf496277eda78493262a1255f245d5c1d97"
ASSET_BASE = "experimentos/balance_nuevo/techniques_kaggle_heavy_v0_4/assets"
EXPECTED_ZIP_SHA = "dd4032c12a574d071411a505565f970473baaa84c17d4302435a9a206938d63b"
EXPECTED_MANIFEST_SHA = "85465dd566690674b94eab91bdd0aebf1053aef76d8373335c5c94aada08bdf6"
PARTS = [f"TECHNIQUES_KAGGLE_HEAVY_INPUTS_V01_1_HOTFIX.part{i:03d}.b64" for i in range(1, 6)]

def digest(raw):
    return hashlib.sha256(raw).hexdigest()

def run_git(*args, cwd=None):
    cmd = ["git", *args]
    print("[GIT]", " ".join(cmd), flush=True)
    try:
        p = subprocess.run(
            cmd,
            cwd=cwd,
            text=True,
            stdout=subprocess.PIPE,
            stderr=None,  # deja visible el progreso de clone/checkout en Kaggle
            check=False,
            timeout=900,
        )
    except subprocess.TimeoutExpired as e:
        raise RuntimeError("GIT_ASSET_BOOTSTRAP_TIMEOUT_900S") from e
    if p.returncode:
        raise RuntimeError("GIT_ASSET_BOOTSTRAP_FAILED:returncode=" + str(p.returncode))
    stdout = (p.stdout or "").strip()
    if stdout:
        print(stdout, flush=True)
    print("[GIT OK]", args[0] if args else "git", flush=True)
    return stdout
def authenticate(root):
    manifest = root / "MANIFEST_SHA256.txt"
    if digest(manifest.read_bytes()) != EXPECTED_MANIFEST_SHA:
        raise RuntimeError("INPUT_PACKAGE_AUTHORITY_MISMATCH")
    names = set()
    for line in manifest.read_text(encoding="utf-8").splitlines():
        expected, name = line.split("  ", 1)
        if name in names or "/" in name or "\\" in name or ":" in name:
            raise RuntimeError("INVALID_INPUT_MANIFEST")
        names.add(name)
        if digest((root / name).read_bytes()) != expected:
            raise RuntimeError("INPUT_HASH_MISMATCH:" + name)
    actual = {p.name for p in root.iterdir() if p.is_file()} - {"MANIFEST_SHA256.txt"}
    if actual != names:
        raise RuntimeError("INPUT_FILE_SET_MISMATCH")
    return root

print("[BOOT 1/6] Python:", sys.version.split()[0], flush=True)
print("[BOOT 2/6] Kaggle input root:", INPUT_ROOT, flush=True)
checkpoint_names = sorted(
    str(p) for p in INPUT_ROOT.rglob("*")
    if "TECHNIQUES_HEAVY_CHECKPOINT" in p.name.upper()
)
print("[BOOT 2/6] Name-matched checkpoint inputs found:", len(checkpoint_names), flush=True)
for p in checkpoint_names[:10]:
    print("  -", p, flush=True)
print("[BOOT 2/6] Mounted dataset roots:", flush=True)
for p in sorted(INPUT_ROOT.iterdir()) if INPUT_ROOT.exists() else []:
    print("  -", p, flush=True)
print("[BOOT 3/6] Fetching authenticated lab assets from GitHub...", flush=True)

ASSET_REPO = WORKING / "La-Grulla-Blanca-Kaggle-Assets"
if ASSET_REPO.exists():
    shutil.rmtree(ASSET_REPO)

run_git("clone", "--progress", "--filter=blob:none", "--no-checkout", OFFICIAL_REPO, str(ASSET_REPO))
run_git("checkout", "--detach", ASSET_COMMIT, cwd=ASSET_REPO)

print("[GIT VERIFY] Reading HEAD independently...", flush=True)
try:
    actual_commit = subprocess.check_output(
        ["git", "rev-parse", "HEAD"],
        cwd=ASSET_REPO,
        text=True,
        stderr=subprocess.STDOUT,
        timeout=60,
    ).strip()
except subprocess.TimeoutExpired as e:
    raise RuntimeError("GIT_REV_PARSE_TIMEOUT") from e
print("[GIT VERIFY] actual_commit =", repr(actual_commit), flush=True)
print("[GIT VERIFY] expected_commit =", repr(ASSET_COMMIT), flush=True)
if actual_commit != ASSET_COMMIT:
    raise RuntimeError(f"ASSET_COMMIT_MISMATCH:{actual_commit}!={ASSET_COMMIT}")
print("[GIT VERIFY OK] Exact asset commit matched.", flush=True)

asset_dir = ASSET_REPO / ASSET_BASE
encoded = "".join((asset_dir / p).read_text(encoding="ascii") for p in PARTS)
zip_bytes = base64.b64decode(encoded, validate=True)
if digest(zip_bytes) != EXPECTED_ZIP_SHA:
    raise RuntimeError("GIT_ASSET_ZIP_SHA_MISMATCH")

selected = WORKING / "TECHNIQUES_KAGGLE_HEAVY_INPUTS_V01_1_HOTFIX.zip"
selected.write_bytes(zip_bytes)

INPUTS = WORKING / "TECHNIQUES_LAB_INPUTS_V01"
if INPUTS.exists():
    shutil.rmtree(INPUTS)

with zipfile.ZipFile(selected) as z:
    if z.testzip() is not None:
        raise RuntimeError("CORRUPTED_INPUT_ZIP")
    if any("/" in n or "\\" in n or ":" in n for n in z.namelist()):
        raise RuntimeError("INVALID_INPUT_ZIP_PATH")
    if sum(i.file_size for i in z.infolist()) > 32 * 1024 * 1024:
        raise RuntimeError("INPUT_ZIP_SIZE_LIMIT")
    INPUTS.mkdir()
    for name in z.namelist():
        (INPUTS / name).write_bytes(z.read(name))

print("[BOOT 4/6] Authenticating reconstructed lab package...", flush=True)
authenticate(INPUTS)
print("[BOOT 4/6 OK] Git asset commit authenticated:", ASSET_COMMIT, flush=True)
print("[BOOT 4/6 OK] LAB input package authenticated:", INPUTS, flush=True)


In [ ]:
print("[BOOT 5/6] Locating and validating AGUA S4 resume checkpoint...", flush=True)
import shutil, zipfile, hashlib, json, urllib.request

RESUME_INPUT_ROOT = WORKING / "TECHNIQUES_RESUME_INPUT"
if RESUME_INPUT_ROOT.exists():
    shutil.rmtree(RESUME_INPUT_ROOT)
RESUME_INPUT_ROOT.mkdir(parents=True)

EXPECTED_ROOT = "agua"

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

REQUIRED_CHECKPOINT_MEMBERS = {
    "MANIFEST_SHA256.txt",
    "state.json",
    "authority.json",
    "campaign_configuration.json",
}

def inspect_checkpoint(path):
    try:
        with zipfile.ZipFile(path) as z:
            names = set(z.namelist())
            if not REQUIRED_CHECKPOINT_MEMBERS.issubset(names):
                return None
            if z.testzip() is not None:
                return None
            state = json.loads(z.read("state.json"))
        return {
            "path": Path(path),
            "basename": Path(path).name,
            "sequence": int(state.get("checkpoint_sequence", -1)),
            "fights": int(state.get("total_fights", -1)),
            "root_status": state.get("roots", {}).get(EXPECTED_ROOT),
            "sha256": sha256_file(path),
            "state": state,
        }
    except (zipfile.BadZipFile, json.JSONDecodeError, KeyError, ValueError, OSError):
        return None

all_zips = sorted({p.resolve() for p in INPUT_ROOT.rglob("*.zip")})
print("[CHECKPOINT SCAN] ZIP files mounted:", len(all_zips), flush=True)
candidates = []
for p in all_zips:
    info = inspect_checkpoint(p)
    if info is not None:
        candidates.append(info)
        print("[CHECKPOINT SCAN] valid:", p, "| seq=", info["sequence"], "| fights=", info["fights"], "| agua=", info["root_status"], "| sha=", info["sha256"][:16], flush=True)

matches = [c for c in candidates if c["basename"] == EXPECTED_RESUME_BASENAME and c["sha256"] == EXPECTED_RESUME_SHA256]
if not matches:
    matches = [c for c in candidates if c["sha256"] == EXPECTED_RESUME_SHA256]

if not matches and CHECKPOINT_URL_FALLBACK:
    downloaded = WORKING / EXPECTED_RESUME_BASENAME
    print("[CHECKPOINT DOWNLOAD] Expected mounted checkpoint not found; trying prior Kaggle Output URL...", flush=True)
    try:
        with urllib.request.urlopen(CHECKPOINT_URL_FALLBACK, timeout=120) as src, open(downloaded, "wb") as dst:
            shutil.copyfileobj(src, dst, length=8 * 1024 * 1024)
        info = inspect_checkpoint(downloaded)
        if info is not None:
            candidates.append(info)
            if info["sha256"] == EXPECTED_RESUME_SHA256:
                matches = [info]
        print("[CHECKPOINT DOWNLOAD] downloaded MB:", round(downloaded.stat().st_size / 1024 / 1024, 2), flush=True)
    except Exception as e:
        print("[CHECKPOINT DOWNLOAD] fallback unavailable:", type(e).__name__, str(e)[:300], flush=True)

if not matches:
    seen = [(c["basename"], c["sha256"], c["sequence"], c["root_status"]) for c in candidates]
    raise RuntimeError("EXPECTED_AGUA_S4_CHECKPOINT_NOT_FOUND_OR_SHA_MISMATCH:" + repr(seen[:20]))

hashes = {c["sha256"] for c in matches}
if hashes != {EXPECTED_RESUME_SHA256}:
    raise RuntimeError("AMBIGUOUS_OR_WRONG_AGUA_S4_CHECKPOINT")

selected = sorted(matches, key=lambda c: (c["basename"] != EXPECTED_RESUME_BASENAME, str(c["path"])))[0]
if selected["root_status"] not in ("RUNNING", "COMPLETE"):
    raise RuntimeError("AGUA_S4_WRONG_ROOT_STATUS:" + str(selected["root_status"]))

staged = RESUME_INPUT_ROOT / "TECHNIQUES_HEAVY_CHECKPOINT.zip"
shutil.copy2(selected["path"], staged)
if sha256_file(staged) != EXPECTED_RESUME_SHA256:
    raise RuntimeError("STAGED_AGUA_S4_SHA_MISMATCH")

with zipfile.ZipFile(staged) as z:
    staged_state = json.loads(z.read("state.json"))
    phases = staged_state.get("phases", {})
    complete_techniques = sorted(
        key.split(":", 1)[1]
        for key, val in phases.items()
        if key.startswith("TECHNIQUE:") and isinstance(val, dict) and val.get("status") == "COMPLETE"
    )

expected_completed = {"espejo_luna", "latigazo_marea", "palma_ardiente"}
if not expected_completed.issubset(set(complete_techniques)):
    raise RuntimeError("AGUA_S4_STATE_GUARD_FAIL:missing_completed=" + repr(sorted(expected_completed - set(complete_techniques))))

print("[CHECKPOINT OK] Source:", selected["path"], flush=True)
print("[CHECKPOINT OK] checkpoint_sequence:", staged_state.get("checkpoint_sequence"), flush=True)
print("[CHECKPOINT OK] total_fights:", staged_state.get("total_fights"), flush=True)
print("[CHECKPOINT OK] completed techniques:", complete_techniques, flush=True)
print("[CHECKPOINT OK] Size MB:", round(staged.stat().st_size / 1024 / 1024, 2), flush=True)
print("[CHECKPOINT OK] SHA256:", sha256_file(staged), flush=True)


In [ ]:
print("[BOOT 6/6] Preparing pinned CPU dependencies...", flush=True)
import subprocess
if sys.version_info < (3, 11):
    raise RuntimeError("PYTHON_3_11_OR_NEWER_REQUIRED")

DEPENDENCIES = json.loads(
    (INPUTS / "CAMPAIGN_CONFIG.json").read_text(encoding="utf-8")
)["DEPENDENCIES"]

DEPENDENCY_TARGET = WORKING / "TECHNIQUES_DEPENDENCIES_V01_1"
lock_file = DEPENDENCY_TARGET / "PINNED_DEPENDENCIES.json"

if not lock_file.exists() or json.loads(lock_file.read_text()) != DEPENDENCIES:
    if DEPENDENCY_TARGET.exists():
        shutil.rmtree(DEPENDENCY_TARGET)
    DEPENDENCY_TARGET.mkdir(parents=True, exist_ok=True)
    print("[PIP] Installing isolated pinned dependencies; this can take a few minutes...", flush=True)
    subprocess.run(
        [
            sys.executable, "-m", "pip", "install",
            "--disable-pip-version-check",
            "--quiet",
            "--ignore-installed",
            "--no-warn-script-location",
            "--target", str(DEPENDENCY_TARGET),
        ] + [k + "==" + v for k, v in DEPENDENCIES.items()],
        check=True,
    )
    lock_file.write_text(json.dumps(DEPENDENCIES), encoding="utf-8")

print("[BOOT COMPLETE] Pinned CPU dependencies ready. Simulation can start.", flush=True)


In [ ]:
print(f"[RUNNER START] scope={ENTRYPOINT} mode={RUN_MODE} resume={RESUME}", flush=True)
request = {
    "inputs": str(INPUTS),
    "working": str(WORKING),
    "input_root": str(RESUME_INPUT_ROOT),
    "RUN_MODE": RUN_MODE,
    "resume": RESUME,
    "scope": ENTRYPOINT,
}
request_file = WORKING / "TECHNIQUES_RUN_REQUEST_AGUA_S4_TO_S5.json"
request_file.write_text(json.dumps(request), encoding="utf-8")

environment = dict(os.environ)
environment["PYTHONPATH"] = str(DEPENDENCY_TARGET) + os.pathsep + str(INPUTS)
environment["PYTHONDONTWRITEBYTECODE"] = "1"

command = [sys.executable, "-u", "-B", str(INPUTS / "campaign_entrypoint.py"), "--run", str(request_file)]
with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=environment) as process:
    for line in process.stdout:
        print(line, end="", flush=True)
    status = process.wait()

if status:
    raise RuntimeError("CAMPAIGN_FAILED: inspect the exact evidence above; AGUA S4 checkpoint remains untouched")
print("Agua S4 continuation finished/yielded cleanly.")


In [ ]:
# Compact the checkpoint conservatively:
# only raw finalist batches belonging to techniques already marked COMPLETE are removed.
import re, zipfile, hashlib, json

ARTIFACTS = WORKING / "TECHNIQUES_CAMPAIGN_V01"
SOURCE_CP = ARTIFACTS / "TECHNIQUES_HEAVY_CHECKPOINT.zip"
COMPACT_CP = ARTIFACTS / "AGUA_S5_TECHNIQUES_HEAVY_CHECKPOINT.zip"

def _sha(raw):
    return hashlib.sha256(raw).hexdigest()

def compact_checkpoint(src, dst):
    with zipfile.ZipFile(src, "r") as zin:
        state = json.loads(zin.read("state.json"))
        phases = state.get("phases", {})
        completed = {
            key.split(":", 1)[1]
            for key, val in phases.items()
            if key.startswith("TECHNIQUE:")
            and isinstance(val, dict)
            and val.get("status") == "COMPLETE"
        }

        kept = []
        removed = []
        for info in zin.infolist():
            name = info.filename
            if name in {"MANIFEST_SHA256.txt", "COMPACTION_REPORT.json"}:
                continue
            parts = name.split("/")
            prune = (
                len(parts) >= 7
                and parts[0] == "roots"
                and parts[2] in completed
                and parts[3] == "finalists"
                and re.fullmatch(r"R\d+", parts[5] or "") is not None
            )
            if prune:
                removed.append((name, info.compress_size, info.file_size))
            else:
                kept.append((name, zin.read(name)))

    report = {
        "status": "SAFE_COMPLETED_TECHNIQUE_RAW_COMPACTION",
        "completed_techniques_detected": sorted(completed),
        "removed_files": len(removed),
        "removed_uncompressed_bytes": sum(x[2] for x in removed),
        "removed_original_zip_compressed_bytes": sum(x[1] for x in removed),
        "policy": (
            "Only raw finalist R<replicates>/ batches of TECHNIQUE:<id>=COMPLETE are removed. "
            "Summary JSON, human shortlist, studies, baselines, authority and state are retained."
        ),
    }
    kept.append((
        "COMPACTION_REPORT.json",
        json.dumps(report, sort_keys=True, separators=(",", ":"), ensure_ascii=False).encode("utf-8"),
    ))

    manifest = [f"{_sha(data)}  {name}" for name, data in sorted(kept)]
    pending = dst.with_suffix(dst.suffix + ".pending")

    with zipfile.ZipFile(pending, "w", zipfile.ZIP_DEFLATED, compresslevel=3) as zout:
        for name, data in sorted(kept):
            zout.writestr(name, data)
        zout.writestr("MANIFEST_SHA256.txt", "\n".join(manifest) + "\n")

    pending.replace(dst)

    # Full integrity verification.
    with zipfile.ZipFile(dst) as z:
        names = z.namelist()
        if len(names) != len(set(names)) or z.testzip() is not None:
            raise RuntimeError("COMPACT_CHECKPOINT_CORRUPTED")
        listed = {}
        for line in z.read("MANIFEST_SHA256.txt").decode("utf-8").splitlines():
            h, name = line.split("  ", 1)
            if name in listed or name not in names or _sha(z.read(name)) != h:
                raise RuntimeError("COMPACT_CHECKPOINT_MANIFEST_MISMATCH:" + name)
            listed[name] = h
        if set(listed) != set(names) - {"MANIFEST_SHA256.txt"}:
            raise RuntimeError("COMPACT_CHECKPOINT_FILESET_MISMATCH")

    return report

if not SOURCE_CP.is_file():
    raise RuntimeError("CAMPAIGN_DID_NOT_PRODUCE_CHECKPOINT")

report = compact_checkpoint(SOURCE_CP, COMPACT_CP)

print("Checkpoint compacted safely.")
print("Completed techniques compacted:", report["completed_techniques_detected"])
print("Removed raw finalist files:", report["removed_files"])
print("Original MB:", round(SOURCE_CP.stat().st_size / 1024 / 1024, 2))
print("Compact MB:", round(COMPACT_CP.stat().st_size / 1024 / 1024, 2))
print("Compact SHA256:", hashlib.sha256(COMPACT_CP.read_bytes()).hexdigest())

from IPython.display import display, FileLink

print("\nDOWNLOAD FOR NEXT SESSION (AGUA S5):")
display(FileLink(os.path.relpath(COMPACT_CP, Path.cwd())))

root_result = ARTIFACTS / "RESULTS_AGUA_HEAVY.zip"
if root_result.is_file():
    print("\nAGUA COMPLETE — DOWNLOAD RESULT (the S4 checkpoint remains as archival/resume evidence):")
    display(FileLink(os.path.relpath(root_result, Path.cwd())))
else:
    print("\nAGUA is not complete yet. Use the compact checkpoint above in the next session.")


In [ ]:
from pathlib import Path

print("\n=== AGUA S4 → S5 OUTPUT SUMMARY ===")
for p in [
    Path("/kaggle/working/TECHNIQUES_CAMPAIGN_V01/AGUA_S5_TECHNIQUES_HEAVY_CHECKPOINT.zip"),
    Path("/kaggle/working/TECHNIQUES_CAMPAIGN_V01/RESULTS_AGUA_HEAVY.zip"),
]:
    if p.is_file():
        print(f"READY: {p}  ({p.stat().st_size/1024/1024:.2f} MB)")
    else:
        print(f"NOT PRESENT: {p}")
